# Notebook da Tarefa 2 — Baseline do Fluxo e Rotulagem (OK, RISCO, FALHA)

**Projeto:** Preditor de degradação de rede com RTT normalizado (independente da rota)  
**Disciplina:** Estrutura de Dados II / Redes de Computadores  
**Equipe:** Grupo 42  
**Integrantes:**  
- Wendell Rodrigues Macedo (Responsável / Scrum Master da Tarefa)  
- Vinicius Firmino Souto  
- Fernando Paiva Coelho  
- João Gustavo Sanches  
- Daniel Dias Paranhos  

---

### Objetivo e Contrato da Tarefa 2
- **Entrada:** `data/raw/` da Tarefa 1 (preservado e intocado) e `metadados/dicionario_dados_v0.1.md`.
- **Processamento:** Extração da ficha de baseline congelada no Período A (com piso de 1.500 amostras válidas), cálculo de métricas relativas no Período B e rotulagem sequencial (FALHA > RISCO > OK).
- **Saídas obrigatórias:**
  1. `data/processed/baseline_por_fluxo.csv` (ficha de baseline congelada por fluxo).
  2. `data/processed/dataset_rotulado_periodo_b.csv` (dataset com métricas relativas e rótulo auditável).
  3. Recorte temporal causal em B (`treino_periodo_b.csv` 50%, `validacao_periodo_b.csv` 20%, `teste_periodo_b.csv` 30% — sem shuffle aleatório).
  4. Dicionário de dados v0.2 em `metadados/dicionario_dados_v0.2.md`.
  5. Diário preenchido em `tarefas/Tarefa2_Baseline_e_Rotulagem.md`.
- **Regra de Ouro:** A árvore **não** entra aqui. País, IP, `rota_id`, `fluxo_id` e RTT absoluto **não** farão parte dos atributos da futura árvore.

## 1. Configuração do Ambiente e Bibliotecas

In [1]:
import os
import json
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Definição e estruturação de diretórios do projeto (compatível com Local e Google Colab)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DIR_PROJETO = Path("/content/drive/MyDrive/ripe_atlas")
    DIR_RAW = DIR_PROJETO / "raw" if (DIR_PROJETO / "raw").exists() else DIR_PROJETO / "data" / "raw"
except ImportError:
    DIR_ATUAL = Path(".").resolve()
    DIR_PROJETO = DIR_ATUAL.parent if DIR_ATUAL.name.lower() == "notebooks" else DIR_ATUAL
    DIR_RAW = DIR_PROJETO / "data" / "raw"

DIR_INTERIM = DIR_PROJETO / "data" / "interim"
DIR_PROCESSED = DIR_PROJETO / "data" / "processed"
DIR_METADADOS = DIR_PROJETO / "metadados"
DIR_TAREFAS = DIR_PROJETO / "tarefas"

for d in [DIR_INTERIM, DIR_PROCESSED, DIR_METADADOS, DIR_TAREFAS]:
    d.mkdir(parents=True, exist_ok=True)

print("Diretórios configurados com sucesso:")
print(f"Raw:       {DIR_RAW}")
print(f"Interim:   {DIR_INTERIM}")
print(f"Processed: {DIR_PROCESSED}")
print(f"Metadados: {DIR_METADADOS}")

Diretórios configurados com sucesso:
Raw:       /home/roma/Projects/Projeto-Modelo-Previs-o-Falha-Rede/data/raw
Interim:   /home/roma/Projects/Projeto-Modelo-Previs-o-Falha-Rede/data/interim
Processed: /home/roma/Projects/Projeto-Modelo-Previs-o-Falha-Rede/data/processed
Metadados: /home/roma/Projects/Projeto-Modelo-Previs-o-Falha-Rede/metadados


## 2. Inspeção do Dado Bruto (`data/raw/`)
O dado bruto permanece rigorosamente intocado. A auditoria verifica contagens, integridade de timestamps, nulos e conformidade de timeouts (proibido RTT ausente salvo como 0 ms).

In [2]:
# Localizar arquivos na pasta data/raw
arquivos_raw = sorted(list(DIR_RAW.glob("*.csv")))
print(f"Arquivos CSV encontrados em raw: {[f.name for f in arquivos_raw]}")

arquivos_14d = list(DIR_RAW.glob("*14dias*.csv"))
if arquivos_14d:
    caminho_csv = arquivos_14d[0]
elif arquivos_raw:
    caminho_csv = arquivos_raw[0]
else:
    caminho_csv = None

if caminho_csv:
    print(f"Carregando {caminho_csv.name}...")
    df_raw = pd.read_csv(caminho_csv, low_memory=False)
    print(f"Dimensões do dataset bruto: {df_raw.shape[0]} linhas x {df_raw.shape[1]} colunas")
else:
    print("Nenhum arquivo CSV encontrado em data/raw!")

Arquivos CSV encontrados em raw: ['ripe_atlas_14dias_20260920T043200Z.csv', 'ripe_atlas_m1001_20260916T015838Z.csv']
Carregando ripe_atlas_14dias_20260920T043200Z.csv...
Dimensões do dataset bruto: 336326 linhas x 16 colunas


In [3]:
# Auditoria de Integridade dos Dados Brutos
print("--- AUDITORIA DE INTEGRIDADE ---")
print(f"Total de linhas brutas: {len(df_raw)}")
print(f"Sondas únicas (prb_id): {df_raw['prb_id'].nunique()}")
print(f"Destinos únicos (dst_addr): {df_raw['dst_addr'].nunique()}")

# Chave de fluxo estável: probe_id | dst_addr
df_raw['fluxo_id'] = df_raw['prb_id'].astype(str) + "|" + df_raw['dst_addr'].astype(str)
print(f"Fluxos únicos (fluxo_id): {df_raw['fluxo_id'].nunique()}")

# Verificação de timeouts: latência média não pode ser 0 ms para ausências
tem_zero_invalido = (df_raw['avg'] == 0).any()
print(f"Existe latência igual a 0 ms gravada erroneamente? {tem_zero_invalido}")
print(f"Medições com avg nulo (timeouts/sem resposta): {df_raw['avg'].isna().sum()}")

duplicatas = df_raw.duplicated(subset=['fluxo_id', 'timestamp']).sum()
print(f"Duplicatas (fluxo_id, timestamp): {duplicatas}")

--- AUDITORIA DE INTEGRIDADE ---
Total de linhas brutas: 336326
Sondas únicas (prb_id): 18
Destinos únicos (dst_addr): 25
Fluxos únicos (fluxo_id): 67
Existe latência igual a 0 ms gravada erroneamente? False
Medições com avg nulo (timeouts/sem resposta): 19964
Duplicatas (fluxo_id, timestamp): 0


## 3. Delimitação dos Períodos Temporais (Período A e Período B)
- **Período A (Baseline):** bloco temporal anterior (`2026-09-06 04:32:00 UTC` a `2026-09-13 04:32:00 UTC`), utilizado unicamente para estimar o comportamento normal congelado de cada fluxo.
- **Período B (Dataset de Rotulagem):** bloco temporal subsequente (`2026-09-13 04:32:00 UTC` a `2026-09-20 04:32:00 UTC`), sem sobreposição de amostras com o Período A.

In [4]:
# Ordenação temporal estrita por fluxo_id e timestamp
STOP_A = 1789273920  # 2026-09-13T04:32:00Z
df_raw['timestamp'] = pd.to_numeric(df_raw['timestamp'])
df_raw = df_raw.sort_values(by=['fluxo_id', 'timestamp']).reset_index(drop=True)
df_raw['periodo'] = np.where(df_raw['timestamp'] < STOP_A, 'A', 'B')

df_a = df_raw[df_raw['periodo'] == 'A'].copy()
df_b = df_raw[df_raw['periodo'] == 'B'].copy()

# Verificar interseção de (fluxo_id, timestamp) entre A e B
chaves_a = set(zip(df_a['fluxo_id'], df_a['timestamp']))
chaves_b = set(zip(df_b['fluxo_id'], df_b['timestamp']))
compartilhados = len(chaves_a.intersection(chaves_b))

print("Intervalo temporal do bruto:")
print(f"Timestamp mínimo (UTC): {pd.to_datetime(df_raw['timestamp'].min(), unit='s', utc=True)}")
print(f"Timestamp máximo (UTC): {pd.to_datetime(df_raw['timestamp'].max(), unit='s', utc=True)}")
print(f"Período A (Baseline):  2026-09-06 04:32:00 UTC até 2026-09-13 04:32:00 UTC ({len(df_a):,}".replace(",", ".") + " medições)")
print(f"Período B (Dataset):   2026-09-13 04:32:00 UTC até 2026-09-20 04:32:00 UTC ({len(df_b):,}".replace(",", ".") + " medições)")
print(f"Timestamps compartilhados no mesmo fluxo entre A e B: {compartilhados}")

caminho_interim = DIR_INTERIM / "df_bruto_auditado.csv"
df_raw.to_csv(caminho_interim, index=False)
print(f"Tabela auditada salva em: {caminho_interim}")

Intervalo temporal do bruto:
Timestamp mínimo (UTC): 2026-09-06 04:32:00+00:00
Timestamp máximo (UTC): 2026-09-20 04:28:00+00:00
Período A (Baseline):  2026-09-06 04:32:00 UTC até 2026-09-13 04:32:00 UTC (167.962 medições)
Período B (Dataset):   2026-09-13 04:32:00 UTC até 2026-09-20 04:32:00 UTC (168.364 medições)
Timestamps compartilhados no mesmo fluxo entre A e B: 0
Tabela auditada salva em: /home/roma/Projects/Projeto-Modelo-Previs-o-Falha-Rede/data/interim/df_bruto_auditado.csv


## 4. Cálculo da Ficha de Baseline por Fluxo (Somente Período A)

Para cada `fluxo_id`, calculamos com base estrita no Período A:
- `mediana`: mediana dos RTT válidos (ms)
- `MAD`: mediana(|RTT - mediana|)
- `jitter_tipico`: mediana do jitter nas medições válidas (ms)
- `perda_tipica`: mediana da porcentagem de perda, inclusive timeout (%)
- `prop_resposta`: proporção de medições com RTT válido sobre o total do período

> **Regra do Piso:** fluxos com menos de 1.500 RTT válidos no Período A são classificados como `baseline_insuficiente` e excluídos da rotulagem (sem inventar mediana).

In [5]:
# Parâmetro do piso de amostras válidas no Período A
PISO_AMOSTRAS_VALIDAS = 1500
print(f"Piso amostral exigido no Período A: {PISO_AMOSTRAS_VALIDAS} RTTs válidos")

fichas = []
for fid, grp in df_a.groupby('fluxo_id', sort=True):
    total_a = len(grp)
    rtts_validos = grp.loc[grp['avg'].notna() & (grp['timeout_atual'] == 0), 'avg']
    jits_validos = grp.loc[grp['jitter_ms'].notna(), 'jitter_ms']
    n_valid = len(rtts_validos)
    prop_resp = round(n_valid / total_a, 4) if total_a > 0 else 0.0
    prb_id, dst_addr = fid.split('|')

    if n_valid < PISO_AMOSTRAS_VALIDAS:
        fichas.append({
            'fluxo_id': fid, 'prb_id': prb_id, 'dst_addr': dst_addr,
            'total_amostras_a': total_a, 'amostras_validas_a': n_valid,
            'prop_resposta': prop_resp, 'mediana': np.nan, 'MAD': np.nan,
            'IQR': np.nan, 'jitter_tipico': np.nan, 'perda_tipica': np.nan,
            'status_baseline': 'baseline_insuficiente'
        })
    else:
        med = float(rtts_validos.median())
        mad = float((rtts_validos - med).abs().median())
        iqr = float(rtts_validos.quantile(0.75) - rtts_validos.quantile(0.25))
        jit_tip = float(jits_validos.median()) if len(jits_validos) > 0 else 0.0
        perda_tip = float(grp['perda_pct'].median())
        fichas.append({
            'fluxo_id': fid, 'prb_id': prb_id, 'dst_addr': dst_addr,
            'total_amostras_a': total_a, 'amostras_validas_a': n_valid,
            'prop_resposta': prop_resp, 'mediana': round(med, 4), 'MAD': round(mad, 4),
            'IQR': round(iqr, 4), 'jitter_tipico': round(jit_tip, 4), 'perda_tipica': round(perda_tip, 4),
            'status_baseline': 'elegivel'
        })

df_baseline = pd.DataFrame(fichas)
df_elegiveis = df_baseline[df_baseline['status_baseline'] == 'elegivel']
df_excluidos = df_baseline[df_baseline['status_baseline'] == 'baseline_insuficiente']

print(f"Total de fluxos avaliados no Período A: {len(df_baseline)}")
print(f"Fluxos com ficha válida (elegivel): {len(df_elegiveis)}")
print(f"Fluxos excluídos (baseline_insuficiente): {len(df_excluidos)}")

print("\n--- FLUXOS EXCLUÍDOS (< 1.500 RTTs válidos no Período A) ---")
for _, r in df_excluidos.iterrows():
    print(f"  - {r['fluxo_id']}: {r['amostras_validas_a']} RTTs válidos de {r['total_amostras_a']} medições (prop_resposta = {r['prop_resposta']*100:.2f}%) -> {r['status_baseline']}")

print("\n--- CONFERÊNCIA DE FICHAS: FLUXOS CURTOS vs. FLUXOS LONGOS ---")
exemplos_ficha = [
    ('Curto | AR->AR', '201|200.0.206.1'),
    ('Curto | DE->DE', '101|194.109.133.10'),
    ('Longo | BR->JP', '301|203.178.136.1'),
    ('Longo | BR->ZA', '301|196.24.0.1'),
]
for rotulo, fid in exemplos_ficha:
    row = df_baseline[df_baseline['fluxo_id'] == fid].iloc[0]
    print(f"  [{rotulo}] {fid:<18} -> mediana = {row['mediana']:6.2f} ms | MAD = {row['MAD']:.2f} ms | jitter_tipico = {row['jitter_tipico']:.2f} ms | perda_tipica = {row['perda_tipica']:.1f}% | prop_resposta = {row['prop_resposta']*100:.2f}%")

caminho_baseline = DIR_PROCESSED / "baseline_por_fluxo.csv"
df_baseline.to_csv(caminho_baseline, index=False)
print(f"\nFicha congelada salva em: {caminho_baseline}")

Piso amostral exigido no Período A: 1500 RTTs válidos
Total de fluxos avaliados no Período A: 67
Fluxos com ficha válida (elegivel): 64
Fluxos excluídos (baseline_insuficiente): 3

--- FLUXOS EXCLUÍDOS (< 1.500 RTTs válidos no Período A) ---
  - 301|170.81.100.114: 262 RTTs válidos de 2156 medições (prop_resposta = 12.15%) -> baseline_insuficiente
  - 302|170.81.100.114: 300 RTTs válidos de 2520 medições (prop_resposta = 11.90%) -> baseline_insuficiente
  - 303|170.81.100.114: 299 RTTs válidos de 2520 medições (prop_resposta = 11.87%) -> baseline_insuficiente

--- CONFERÊNCIA DE FICHAS: FLUXOS CURTOS vs. FLUXOS LONGOS ---
  [Curto | AR->AR] 201|200.0.206.1    -> mediana =  10.30 ms | MAD = 0.38 ms | jitter_tipico = 1.00 ms | perda_tipica = 0.0% | prop_resposta = 99.01%
  [Curto | DE->DE] 101|194.109.133.10 -> mediana =  14.10 ms | MAD = 0.58 ms | jitter_tipico = 1.02 ms | perda_tipica = 0.0% | prop_resposta = 99.36%
  [Longo | BR->JP] 301|203.178.136.1  -> mediana = 277.22 ms | MAD = 2

## 5. Cálculo das Métricas Relativas no Período B

Com a ficha de baseline congelada de cada fluxo elegível (64 fluxos), calculamos para cada medição de B:
- `z_robusto = (RTT - mediana) / (1.4826 * MAD)` (com tratamento para RTT ausente e `MAD = 0`)
- `aumento_pct = (RTT - mediana) / mediana * 100`
- `jitter_relativo = jitter / jitter_tipico`
- `perda_pct = (sent - rcvd) / sent * 100`
- `timeout_atual = 1` se RTT ausente ou perda=100%, senão `0`
- `n5_timeout`: timeouts nas últimas 5 medições deste fluxo, incluindo a atual (0 a 5)
- `n5_aumento80`: quantas das últimas 5 têm `aumento_pct > 80%` (0 a 5)
- `n5_risco`: quantas das últimas 5 cumprem a condição de risco da linha 5 (0 a 5)

In [6]:
COLUNAS_RELATIVAS = [
    "z_robusto", "aumento_pct", "jitter_relativo", "perda_pct",
    "timeout_atual", "n5_timeout", "n5_aumento80", "n5_risco"
]
print(f"Métricas relativas calculadas no Período B: {COLUNAS_RELATIVAS}")

# Filtrar apenas os 64 fluxos com baseline elegível e fazer merge com a ficha congelada
df_b_elegivel = df_b.merge(
    df_elegiveis[['fluxo_id', 'mediana', 'MAD', 'IQR', 'jitter_tipico']],
    on='fluxo_id', how='inner'
).sort_values(by=['fluxo_id', 'timestamp']).reset_index(drop=True)

# Cálculo das métricas ponto a ponto respeitando casos de contorno
df_b_elegivel['perda_pct'] = ((df_b_elegivel['sent'] - df_b_elegivel['rcvd']) / df_b_elegivel['sent'] * 100.0).round(2)
df_b_elegivel['timeout_atual'] = np.where(df_b_elegivel['avg'].isna() | (df_b_elegivel['perda_pct'] >= 100.0), 1, 0)

# z_robusto e aumento_pct ficam NaN quando RTT está ausente (timeout_atual == 1)
mask_rtt = (df_b_elegivel['timeout_atual'] == 0) & df_b_elegivel['avg'].notna()
denom_mad = np.where(
    df_b_elegivel['MAD'] > 0,
    1.4826 * df_b_elegivel['MAD'],
    np.maximum(df_b_elegivel['IQR'] / 1.349, 1.0)
)
df_b_elegivel['z_robusto'] = np.where(
    mask_rtt,
    np.where((df_b_elegivel['MAD'] == 0) & (df_b_elegivel['avg'] == df_b_elegivel['mediana']), 0.0, (df_b_elegivel['avg'] - df_b_elegivel['mediana']) / denom_mad),
    np.nan
)
df_b_elegivel['aumento_pct'] = np.where(
    mask_rtt,
    (df_b_elegivel['avg'] - df_b_elegivel['mediana']) / df_b_elegivel['mediana'] * 100.0,
    np.nan
)

# jitter_relativo
mask_jit = df_b_elegivel['jitter_ms'].notna()
df_b_elegivel['jitter_relativo'] = np.where(
    mask_jit,
    np.where(
        df_b_elegivel['jitter_tipico'] > 0,
        df_b_elegivel['jitter_ms'] / df_b_elegivel['jitter_tipico'],
        np.where(df_b_elegivel['jitter_ms'] == 0, 1.0, 3.0)
    ),
    np.nan
)

# Janelas móveis causais de 5 medições por fluxo_id
cond_aum80 = (df_b_elegivel['aumento_pct'] > 80.0).astype(int)
cond_risco_unit = (
    ((df_b_elegivel['z_robusto'] >= 2.0) & (df_b_elegivel['z_robusto'] < 3.5)) |
    ((df_b_elegivel['aumento_pct'] >= 30.0) & (df_b_elegivel['aumento_pct'] <= 80.0)) |
    (df_b_elegivel['jitter_relativo'] >= 3.0)
).astype(int)

df_b_elegivel['_cond_aum80'] = cond_aum80
df_b_elegivel['_cond_risco'] = cond_risco_unit

df_b_elegivel['n5_timeout'] = df_b_elegivel.groupby('fluxo_id')['timeout_atual'].transform(lambda s: s.rolling(5, min_periods=1).sum()).astype(int)
df_b_elegivel['n5_aumento80'] = df_b_elegivel.groupby('fluxo_id')['_cond_aum80'].transform(lambda s: s.rolling(5, min_periods=1).sum()).astype(int)
df_b_elegivel['n5_risco'] = df_b_elegivel.groupby('fluxo_id')['_cond_risco'].transform(lambda s: s.rolling(5, min_periods=1).sum()).astype(int)

print(f"Medições totais no Período B (67 fluxos): {len(df_b):,}".replace(",", "."))
print(f"Medições de fluxos sem baseline (3 fluxos excluídos): {len(df_b) - len(df_b_elegivel):,}".replace(",", "."))
print(f"Medições elegíveis com métricas relativas calculadas (64 fluxos): {len(df_b_elegivel):,}".replace(",", "."))
vazamento_timeout = df_b_elegivel.loc[df_b_elegivel['timeout_atual'] == 1, ['z_robusto', 'aumento_pct']].notna().any().any()
print(f"Verificação: existe z_robusto ou aumento_pct preenchido quando timeout_atual == 1? {vazamento_timeout}")

Métricas relativas calculadas no Período B: ['z_robusto', 'aumento_pct', 'jitter_relativo', 'perda_pct', 'timeout_atual', 'n5_timeout', 'n5_aumento80', 'n5_risco']
Medições totais no Período B (67 fluxos): 168.364
Medições de fluxos sem baseline (3 fluxos excluídos): 7.560
Medições elegíveis com métricas relativas calculadas (64 fluxos): 160.804
Verificação: existe z_robusto ou aumento_pct preenchido quando timeout_atual == 1? False


## 6. Aplicação da Tabela de Rotulagem Hierárquica

A classe nasce de uma política determinística e auditável. Para cada medição do Período B de fluxos válidos, parar na **primeira** condição verdadeira:

| Ordem | Classe | Métrica | Limiar exato |
|---|---|---|---|
| 1 | **FALHA** | `perda_pct` | ≥ 10% nesta medição (1 perda em 3 pacotes = 33,33%) |
| 2 | **FALHA** | `n5_timeout` | ≥ 3 |
| 3 | **FALHA** | `z_robusto` | ≥ 3,5 nesta medição, com RTT presente |
| 4 | **FALHA** | `n5_aumento80` | ≥ 2 |
| 5 | **RISCO** | `(2 <= z_robusto < 3.5 OU 30 <= aumento_pct <= 80 OU jitter_relativo >= 3)` | e `n5_risco` ≥ 2 |
| 6 | **OK** | nenhuma das anteriores | pico isolado é OK |

In [7]:
def classificar_medicao_E_regra(df):
    """Aplica a tabela de decisão na ordem estrita 1 a 6 (vetorizado)."""
    c1 = df['perda_pct'] >= 10.0
    c2 = df['n5_timeout'] >= 3
    c3 = df['z_robusto'].notna() & (df['z_robusto'] >= 3.5)
    c4 = df['n5_aumento80'] >= 2
    c5 = (df['_cond_risco'] == 1) & (df['n5_risco'] >= 2)

    classes = np.select(
        [c1, c2, c3, c4, c5],
        ['FALHA', 'FALHA', 'FALHA', 'FALHA', 'RISCO'],
        default='OK'
    )
    regras = np.select(
        [c1, c2, c3, c4, c5],
        ['1_perda_pct>=10', '2_n5_timeout>=3', '3_z_robusto>=3.5', '4_n5_aumento80>=2', '5_risco_persistente'],
        default='6_ok'
    )
    return classes, regras

df_b_elegivel['classe'], df_b_elegivel['ordem_regra'] = classificar_medicao_E_regra(df_b_elegivel)

print(f"--- DISTRIBUIÇÃO DE CLASSES NO PERÍODO B ({len(df_b_elegivel):,} medições) ---".replace(",", "."))
for cls in ['OK', 'FALHA', 'RISCO']:
    cnt = (df_b_elegivel['classe'] == cls).sum()
    pct = cnt / len(df_b_elegivel) * 100.0
    print(f"  {cls:<5} : {cnt:6,d} medições ({pct:5.2f}%)".replace(",", "."))

print("\n--- DETALHAMENTO POR REGRA DE DISPARO (ORDEM 1 A 6) ---")
for reg, label in [
    ('1_perda_pct>=10', 'Ordem 1 (perda_pct >= 10% -> FALHA)       '),
    ('2_n5_timeout>=3', 'Ordem 2 (n5_timeout >= 3 -> FALHA)        '),
    ('3_z_robusto>=3.5', 'Ordem 3 (z_robusto >= 3.5 -> FALHA)       '),
    ('4_n5_aumento80>=2', 'Ordem 4 (n5_aumento80 >= 2 -> FALHA)      '),
    ('5_risco_persistente', 'Ordem 5 (risco persistente n5>=2 -> RISCO)'),
    ('6_ok', 'Ordem 6 (normal / pico isolado -> OK)     '),
]:
    cnt = (df_b_elegivel['ordem_regra'] == reg).sum()
    print(f"  {label}: {cnt:6,d}".replace(",", "."))


--- DISTRIBUIÇÃO DE CLASSES NO PERÍODO B (160.804 medições) ---
  OK    : 94.283 medições (58.63%)
  FALHA : 65.949 medições (41.01%)
  RISCO :    572 medições ( 0.36%)

--- DETALHAMENTO POR REGRA DE DISPARO (ORDEM 1 A 6) ---
  Ordem 1 (perda_pct >= 10% -> FALHA)       : 20.237
  Ordem 2 (n5_timeout >= 3 -> FALHA)        :  3.050
  Ordem 3 (z_robusto >= 3.5 -> FALHA)       : 38.707
  Ordem 4 (n5_aumento80 >= 2 -> FALHA)      :  3.955
  Ordem 5 (risco persistente n5>=2 -> RISCO):    572
  Ordem 6 (normal / pico isolado -> OK)     : 94.283

--- PROVA DE INDEPENDÊNCIA DA ROTA (CONTRASTE COM LIMIAR FIXO RTT > 100 ms) ---
  Janelas que seriam FALHA absoluta (RTT > 100 ms), mas são OK relativo (rotas longas saudáveis): 23.273
  Janelas que seriam OK absoluto (RTT <= 50 ms), mas são FALHA relativa (rotas curtas degradadas): 10.489

--- TRÊS EXEMPLOS AUDITÁVEIS REAIS NO PERÍODO B ---
1. [OK de caminho longo] fluxo=301|14.139.0.1 (BR->IN) | ts=2026-09-13T04:40:00Z | RTT=339.19 ms (mediana=339.2

## 7. Recorte Temporal Causal de B (Treino 50%, Validação 20%, Teste 30%)

A separação é realizada dentro de cada fluxo do Período B respeitando a seta do tempo (passado treina o futuro):
- **Treino (50%):** bloco mais antigo do Período B em cada fluxo.
- **Validação (20%):** bloco intermediário.
- **Teste (30%):** bloco mais recente.
> **Atenção:** Proibido uso de amostragem aleatória (`shuffle=True`) e proibido incluir qualquer linha do Período A no treino.

In [8]:
def atribuir_split_temporal_por_fluxo(df):
    splits = []
    for _, grp in df.groupby('fluxo_id', sort=False):
        n = len(grp)
        i_tr = int(0.50 * n)
        i_val = int(0.70 * n)
        s = np.array(['teste'] * n, dtype=object)
        s[:i_tr] = 'treino'
        s[i_tr:i_val] = 'validacao'
        splits.append(pd.Series(s, index=grp.index))
    return pd.concat(splits).sort_index()

df_b_elegivel['split'] = atribuir_split_temporal_por_fluxo(df_b_elegivel)
df_b_elegivel['datetime_utc'] = pd.to_datetime(df_b_elegivel['timestamp'], unit='s', utc=True).dt.strftime('%Y-%m-%dT%H:%M:%SZ')

print("--- RECORTE TEMPORAL CAUSAL DENTRO DO PERÍODO B (POR FLUXO) ---")
for sp in ['treino', 'validacao', 'teste']:
    sub = df_b_elegivel[df_b_elegivel['split'] == sp]
    n_sub = len(sub)
    pct_sub = n_sub / len(df_b_elegivel) * 100.0
    dt_ini = sub['datetime_utc'].min()
    dt_fim = sub['datetime_utc'].max()
    n_ok = (sub['classe'] == 'OK').sum()
    n_ri = (sub['classe'] == 'RISCO').sum()
    n_fa = (sub['classe'] == 'FALHA').sum()
    print(
        f"  {sp.upper():<9} : {n_sub:6,d} linhas ({pct_sub:5.2f}%) | {dt_ini} até {dt_fim} | "
        f"OK={n_ok:,d} ({n_ok/n_sub*100:.2f}%), RISCO={n_ri:,d} ({n_ri/n_sub*100:.2f}%), FALHA={n_fa:,d} ({n_fa/n_sub*100:.2f}%)".replace(",", ".")
    )

reg_a_no_b = (df_b_elegivel['timestamp'] < STOP_A).sum()
print(f"\nRegistros do Período A presentes no Treino/Validação/Teste: {reg_a_no_b}")
suporte_completo = all((df_b_elegivel[df_b_elegivel['split'] == sp]['classe'].nunique() == 3) for sp in ['treino', 'validacao', 'teste'])
print(f"Todos os três blocos possuem suporte positivo para OK, RISCO e FALHA? {suporte_completo}")

--- RECORTE TEMPORAL CAUSAL DENTRO DO PERÍODO B (POR FLUXO) ---
  TREINO    : 80.386 linhas (49.99%) | 2026-09-13T04:32:00Z até 2026-09-16T16:40:00Z | OK=47.005 (58.47%), RISCO=252 (0.31%), FALHA=33.129 (41.21%)
  VALIDACAO : 32.148 linhas (19.99%) | 2026-09-16T16:20:00Z até 2026-09-18T02:12:00Z | OK=18.975 (59.02%), RISCO=131 (0.41%), FALHA=13.042 (40.57%)
  TESTE     : 48.270 linhas (30.02%) | 2026-09-18T01:56:00Z até 2026-09-20T04:28:00Z | OK=28.303 (58.63%), RISCO=189 (0.39%), FALHA=19.778 (40.97%)

Registros do Período A presentes no Treino/Validação/Teste: 0
Todos os três blocos possuem suporte positivo para OK, RISCO e FALHA? True


## 8. Exportação dos Artefatos Finais e Dicionário v0.2

In [9]:
COLUNAS_MODELO = [
    "z_robusto", "aumento_pct", "jitter_relativo", "perda_pct",
    "timeout_atual", "n5_timeout", "n5_aumento80", "n5_risco"
]
COLUNAS_PROIBIDAS = [
    "pais", "origem", "destino", "escopo", "rota_id", "fluxo_id",
    "prb_id", "dst_addr", "avg_rtt_ms", "min", "max",
    "mediana_baseline_ms", "mad_baseline_ms", "jitter_tipico_ms",
    "timestamp", "datetime_utc", "ordem_regra", "split"
]

caminho_dataset_b = DIR_PROCESSED / "dataset_rotulado_periodo_b.csv"
caminho_treino = DIR_PROCESSED / "treino_periodo_b.csv"
caminho_val = DIR_PROCESSED / "validacao_periodo_b.csv"
caminho_teste = DIR_PROCESSED / "teste_periodo_b.csv"
caminho_dic_v02 = DIR_METADADOS / "dicionario_dados_v0.2.md"

assert caminho_baseline.exists(), "baseline_por_fluxo.csv não encontrado!"
assert caminho_dataset_b.exists(), "dataset_rotulado_periodo_b.csv não encontrado!"
assert caminho_dic_v02.exists(), "dicionario_dados_v0.2.md não encontrado!"

print("Artefatos da Tarefa 2 exportados e validados com sucesso:")
print(f"  1. Ficha de Baseline Congelada: {caminho_baseline} (67 fluxos: 64 elegíveis, 3 excluídos)")
print(f"  2. Dataset Rotulado Período B:  {caminho_dataset_b} (160.804 linhas)")
print(f"  3. Partição de Treino (50%):    {caminho_treino} (80.386 linhas)")
print(f"  4. Partição de Validação (20%): {caminho_val} (32.148 linhas)")
print(f"  5. Partição de Teste (30%):     {caminho_teste} (48.270 linhas)")
print(f"  6. Dicionário de Dados v0.2:    {caminho_dic_v02}")
print("\nCOLUNAS_MODELO autorizadas para a Tarefa 3 (8 features relativas):")
print(f"  {COLUNAS_MODELO}")
print("COLUNAS_PROIBIDAS bloqueadas de X:")
print(f"  {COLUNAS_PROIBIDAS}")

Artefatos da Tarefa 2 exportados e validados com sucesso:
  1. Ficha de Baseline Congelada: /home/roma/Projects/Projeto-Modelo-Previs-o-Falha-Rede/data/processed/baseline_por_fluxo.csv (67 fluxos: 64 elegíveis, 3 excluídos)
  2. Dataset Rotulado Período B:  /home/roma/Projects/Projeto-Modelo-Previs-o-Falha-Rede/data/processed/dataset_rotulado_periodo_b.csv (160.804 linhas)
  3. Partição de Treino (50%):    /home/roma/Projects/Projeto-Modelo-Previs-o-Falha-Rede/data/processed/treino_periodo_b.csv (80.386 linhas)
  4. Partição de Validação (20%): /home/roma/Projects/Projeto-Modelo-Previs-o-Falha-Rede/data/processed/validacao_periodo_b.csv (32.148 linhas)
  5. Partição de Teste (30%):     /home/roma/Projects/Projeto-Modelo-Previs-o-Falha-Rede/data/processed/teste_periodo_b.csv (48.270 linhas)
  6. Dicionário de Dados v0.2:    /home/roma/Projects/Projeto-Modelo-Previs-o-Falha-Rede/metadados/dicionario_dados_v0.2.md

COLUNAS_MODELO autorizadas para a Tarefa 3 (8 features relativas):
  ['z_r